<a href="https://colab.research.google.com/github/Ishige99/Ishige99/blob/main/faster_whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Set up

## faster-whisperをinstall
!pip install faster-whisper

## Google Driveにアクセスして動画ファイルを取得する
import os
from google.colab import drive
drive.mount('/content/drive')

## 音声ファイルを取得
## TODO: 複数対応可能にする
audio_list = os.listdir("/content/drive/MyDrive/audio/")
audio_file_path = "/content/drive/MyDrive/audio/" + audio_list[0]

In [4]:
# srt file を作成するための関数
def create_srt_file(results, use_faster_whisper, file_name="transcribe"):
    data = []
    with open(f"{file_name}.srt", mode="w") as f:
        for index, _dict in enumerate(results):
            if use_faster_whisper:
              start_time = _dict.start # start
              end_time = _dict.end # end
              text = _dict.text # text
            else:
              start_time = _dict["start"]
              end_time = _dict["end"]
              text = _dict["text"]

            data.append({
            "index": index + 1,
            "start": start_time,
            "end": end_time,
            "text": text})

            # 時、分、秒、ミリ秒に分割
            s_h, s_m, s_s = int(start_time // 3600), int((start_time % 3600) // 60), int(start_time % 60)
            e_h, e_m, e_s = int(end_time // 3600), int((end_time % 3600) // 60), int(end_time % 60)

            # ミリ秒を計算
            s_ms = int((start_time - int(start_time)) * 1000)
            e_ms = int((end_time - int(end_time)) * 1000)

            f.write(f"{index+1}\n")
            f.write(f"{s_h:02}:{s_m:02}:{s_s:02},{s_ms:03} --> {e_h:02}:{e_m:02}:{e_s:02},{e_ms:03}\n")
            f.write(f"{text}\n\n")
    return data

In [5]:
import torch
import gc

def release_model_memory(model):
    """
    指定されたモデルをメモリから削除し、ガーベージコレクションとPyTorchのキャッシュメモリ解放を行う関数。

    Parameters:
    model (torch.nn.Module): メモリから解放するモデルオブジェクト。
    """
    # モデルを削除
    del model

    # ガーベージコレクションを実行してメモリを解放
    gc.collect()

    # PyTorchのキャッシュされたメモリを解放
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [6]:
from faster_whisper import WhisperModel

In [ ]:
# model = WhisperModel("large-v3", device="cuda", compute_type="int8_float16")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")

In [8]:
segments, info = model.transcribe(audio_file_path, beam_size=5)
faster_whisper_data = create_srt_file(results=segments, use_faster_whisper=True, file_name="transcribe_faster_whisper")